[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/22_anomaly_detection/03_evaluation_and_practice.ipynb)

# Bài 03 — Đánh giá và thực hành phát hiện bất thường

**🎯 Mục tiêu**
- Ba chế độ: **không giám sát**, **bán giám sát** (novelty — train trên dữ liệu sạch), **có giám sát** — và điểm mù của từng chế độ.
- Vì sao model có giám sát **bỏ sót kiểu gian lận mới**; khi số nhãn tăng dần thì sao.
- Phát hiện bất thường bằng **sai số tái tạo** (PCA) và khi nào nó thất bại; kết hợp nhiều detector.
- Bất thường theo thời gian: so với ngữ cảnh quá khứ, loại trừ ngữ cảnh đã biết (khuyến mãi, ngày lễ).
- Template model cho phát hiện bất thường.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/22_anomaly_detection

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from sklearn.decomposition import PCA
from sklearn.ensemble import HistGradientBoostingClassifier, IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler

from ml_course.datasets import make_sales_series
from ml_course.visualization import set_style

set_style()
rng = np.random.default_rng(0)

## 1. Dữ liệu: giao dịch thẻ với ba **kiểu** gian lận

Gian lận thật không phải "giao dịch bình thường hơi lệch" mà gồm các **kiểu** (pattern) khác hẳn:
- **A — chiếm tài khoản**: số tiền lớn, xa nhà, thường ở nước ngoài.
- **B — thử thẻ**: rất nhiều giao dịch số tiền rất nhỏ trong 24h.
- **C — thẻ mới, rút nhanh**: thẻ mới mở vài ngày đã chi lớn. **C chỉ xuất hiện ở tập test** — kiểu gian lận mới mà hệ thống chưa từng thấy.

In [ ]:
FEATURES = ["log_amount", "log_distance", "tx_last_24h", "log_card_age", "is_foreign"]


def make_transactions(n_normal, n_per_pattern, patterns, seed):
    r = np.random.default_rng(seed)

    def normal(n):
        return pd.DataFrame({
            "log_amount": r.normal(12.7, 0.9, n),                       # ≈ 330 nghìn VND
            "log_distance": np.log1p(r.exponential(8, n)),
            "tx_last_24h": r.poisson(2, n).astype(float),
            "log_card_age": np.log(r.uniform(30, 3650, n)),
            "is_foreign": (r.random(n) < 0.04).astype(float),
        })

    parts = [normal(n_normal).assign(pattern="bình thường")]
    for p in patterns:
        d, m = normal(n_per_pattern), n_per_pattern
        if p == "A":
            d["log_amount"] = r.normal(15.0, 0.6, m)
            d["log_distance"] = np.log1p(r.exponential(300, m))
            d["is_foreign"] = (r.random(m) < 0.6).astype(float)
        elif p == "B":
            d["log_amount"] = r.normal(9.5, 0.5, m)
            d["tx_last_24h"] = r.poisson(15, m).astype(float)
        elif p == "C":
            d["log_card_age"] = np.log(r.uniform(1, 7, m))
            d["log_amount"] = r.normal(14.0, 0.5, m)
            d["tx_last_24h"] = r.poisson(5, m).astype(float)
        parts.append(d.assign(pattern=p))
    df = pd.concat(parts, ignore_index=True)
    df["is_fraud"] = (df["pattern"] != "bình thường").astype(int)
    return df


train = make_transactions(20000, 60, ["A", "B"], seed=0)
test = make_transactions(20000, 40, ["A", "B", "C"], seed=1)
scaler = StandardScaler().fit(train[FEATURES])
X_tr, X_te = scaler.transform(train[FEATURES]), scaler.transform(test[FEATURES])
y_tr, y_te = train["is_fraud"].to_numpy(), test["is_fraud"].to_numpy()
print("train:", train["pattern"].value_counts().to_dict())
print("test :", test["pattern"].value_counts().to_dict())

## 2. Ba chế độ, ba điểm mù

| Chế độ | Dữ liệu train | Ví dụ |
|---|---|---|
| Không giám sát (outlier detection) | toàn bộ dữ liệu (có lẫn bất thường), **không nhãn** | Isolation Forest trên log hệ thống |
| Bán giám sát (novelty detection) | **chỉ dữ liệu bình thường** đã được xác nhận | Gaussian (bài 01), LOF `novelty=True`, autoencoder |
| Có giám sát | nhãn cho cả hai lớp | phân loại mất cân bằng (chương 19) |

Hệ thống chống gian lận thường duyệt thủ công **top-k** giao dịch đáng ngờ mỗi ngày → đo **recall@200 theo từng kiểu**:

In [ ]:
pca = PCA(3).fit(X_tr[y_tr == 0])
scores = {
    "Isolation Forest (không giám sát)": -IsolationForest(n_estimators=300, random_state=0).fit(X_tr).score_samples(X_te),
    "LOF novelty (chỉ dữ liệu bình thường)": -LocalOutlierFactor(n_neighbors=35, novelty=True).fit(X_tr[y_tr == 0]).score_samples(X_te),
    "PCA sai số tái tạo (chỉ bình thường)": np.sum((X_te - pca.inverse_transform(pca.transform(X_te))) ** 2, axis=1),
    "Logistic Regression (có giám sát)": LogisticRegression(max_iter=2000).fit(X_tr, y_tr).predict_proba(X_te)[:, 1],
    "HistGradientBoosting (có giám sát)": HistGradientBoostingClassifier(random_state=0).fit(X_tr, y_tr).predict_proba(X_te)[:, 1],
}


def recall_at_k_by_pattern(score, k=200):
    top = test["pattern"].to_numpy()[np.argsort(-score)[:k]]
    return {f"recall@{k} kiểu {p}": (top == p).sum() / (test["pattern"] == p).sum() for p in "ABC"}


summary = pd.DataFrame({name: {"PR-AUC": average_precision_score(y_te, s), **recall_at_k_by_pattern(s)}
                        for name, s in scores.items()}).T.round(2)
summary

Đọc bảng:
- **Có giám sát** (LogReg, HGB) bắt rất tốt **A và B** — những kiểu đã có nhãn — nhưng bắt được rất ít **C**: model chỉ học "gian lận trông giống A hoặc B", còn tuổi thẻ (đặc trưng của C) chưa từng phân biệt gian lận ở train.
- **LOF novelty** — học "bình thường trông như thế nào" từ dữ liệu sạch — bắt cả kiểu mới C: bất cứ thứ gì **không giống bình thường** đều đáng ngờ.
- **Isolation Forest** (không cần nhãn, train trên dữ liệu có lẫn gian lận) ở mức trung bình.
- **Sai số tái tạo PCA bỏ sót hoàn toàn A**: các feature bình thường gần như độc lập (không có cấu trúc ít chiều), nên 3 thành phần chính chỉ là 3 hướng "tuỳ ý"; độ lệch của A nằm gọn trong không gian 3 chiều đó → tái tạo tốt → sai số nhỏ. Sai số tái tạo chỉ hiệu quả khi dữ liệu bình thường thật sự **nằm gần một không gian con/đa tạp ít chiều** (ảnh, tín hiệu cảm biến tương quan — chương 34 với autoencoder).

> ⚠️ Kết quả của LOF ở đây đẹp vì dữ liệu chỉ có 5 feature và các kiểu gian lận khác biệt rõ. Với hàng trăm feature, khoảng cách kém ý nghĩa (curse of dimensionality, chương 08) — detector dựa khoảng cách xuống cấp.

## 3. Khi số nhãn tăng dần

Giả sử chỉ có $m$ giao dịch gian lận (A hoặc B) được gán nhãn, kèm tỉ lệ giao dịch bình thường được gán nhãn tương ứng:

In [ ]:
iso_ap = average_precision_score(y_te, scores["Isolation Forest (không giám sát)"])
pos, neg = np.flatnonzero(y_tr == 1), np.flatnonzero(y_tr == 0)
rows = []
for m in [3, 5, 10, 20, 50, len(pos)]:
    aps = []
    for seed in range(5):
        r = np.random.default_rng(seed)
        idx = np.r_[r.choice(pos, m, replace=False), r.choice(neg, int(m * len(neg) / len(pos)), replace=False)]
        lr = LogisticRegression(max_iter=2000, class_weight="balanced").fit(X_tr[idx], y_tr[idx])
        aps.append(average_precision_score(y_te, lr.predict_proba(X_te)[:, 1]))
    rows.append({"số gian lận có nhãn": m, "LogReg PR-AUC (TB)": np.mean(aps), "± std (5 lần lấy mẫu)": np.std(aps),
                 "Isolation Forest PR-AUC": iso_ap})
pd.DataFrame(rows).round(3)

- Với vài chục nhãn trở xuống, model có giám sát **kém hơn** detector không giám sát và **dao động mạnh** theo việc *những giao dịch nào* được gán nhãn.
- Khi số nhãn đủ lớn (≥ 50), nó nhỉnh hơn Isolation Forest — nhưng vẫn **không bắt được kiểu C** (mục 2), dù có thêm bao nhiêu nhãn A/B.

Thực tế thường **kết hợp**: detector không giám sát/novelty tìm ứng viên → chuyên gia xác nhận → tích luỹ nhãn → model có giám sát cho các kiểu đã biết, **song song** với detector bất thường cho kiểu mới (hoặc dùng điểm bất thường làm **feature** cho model có giám sát).

## 4. Kết hợp detector khi không có nhãn

Khi không có nhãn, ta **không biết** detector nào tốt nhất cho dữ liệu của mình. Chuẩn hoá điểm bằng **thứ hạng** rồi lấy trung bình là cách phòng ngừa rủi ro:

In [ ]:
unsup = {k: v for k, v in scores.items() if "có giám sát" not in k}
ens = np.mean([rankdata(s) / len(s) for s in unsup.values()], axis=0)
for name, s in unsup.items():
    print(f"{name:40s} PR-AUC = {average_precision_score(y_te, s):.3f}")
print(f"{'trung bình thứ hạng (3 detector)':40s} PR-AUC = {average_precision_score(y_te, ens):.3f}")

Kết hợp **không vượt** detector tốt nhất (LOF), nhưng tránh được detector tệ nhất (PCA) — đó là mục đích: khi không thể biết trước detector nào hợp, trung bình thứ hạng cho kết quả "an toàn". Nếu có một ít nhãn, dùng chúng để **chọn** hoặc **đánh trọng số** detector (cẩn thận: vài chục nhãn → đánh giá rất nhiễu).

## 5. Bất thường theo thời gian

Với chuỗi thời gian, "bình thường" phụ thuộc **ngữ cảnh** (xu hướng, mùa vụ, khuyến mãi). Ý tưởng:
1. Khử mùa vụ tuần: so mỗi ngày với **trung vị 4 ngày cùng thứ** trong 4 tuần trước (trung vị → một sự cố cũ không làm hỏng mốc so sánh).
2. Robust z-score của tỉ số đó trên cửa sổ 28 ngày **quá khứ** (`.shift(1)` — không rò rỉ).
3. Bỏ cảnh báo vào ngày có **ngữ cảnh đã biết** (khuyến mãi, ngày lễ) — đó là bất thường "có lý do".

In [ ]:
sales = make_sales_series(n_days=730, seed=3)
s = sales.set_index("date")["sales"].copy()
anom_days = rng.choice(np.arange(60, 730), 8, replace=False)
s.iloc[anom_days] *= rng.choice([0.4, 1.9], 8)                       # sự cố: doanh số sụt/vọt bất thường
baseline = pd.concat([s.shift(7 * k) for k in range(1, 5)], axis=1).median(axis=1)
ratio = np.log(s / baseline)
med = ratio.rolling(28, min_periods=14).median().shift(1)
mad = (ratio - med).abs().rolling(28, min_periods=14).median().shift(1)
rz = 0.6745 * (ratio - med) / mad
flag = (rz.abs() > 4).fillna(False).to_numpy()
known_context = (sales["promo"] == 1).to_numpy() | (sales["is_holiday"] == 1).to_numpy()
alert = flag & ~known_context
for name, f in [("mọi cảnh báo", flag), ("bỏ ngày khuyến mãi/lễ", alert)]:
    print(f"{name:22s}: {f.sum():2d} cảnh báo | bắt {f[anom_days].sum()}/8 sự cố")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(s.index, s.values, lw=0.8)
ax.scatter(s.index[anom_days], s.iloc[anom_days], s=90, facecolors="none", edgecolors="k", label="sự cố thật")
ax.scatter(s.index[flag & known_context], s[flag & known_context], color="orange", s=15, label="cảnh báo bị bỏ (khuyến mãi/lễ)")
ax.scatter(s.index[alert], s[alert], color="r", s=15, label="cảnh báo")
ax.legend()
ax.set_title("Robust z-score của log(doanh số / trung vị 4 ngày cùng thứ trước đó)")
plt.show()

Ngày khuyến mãi/lễ vọt doanh số là **bất thường theo thống kê** nhưng **bình thường theo ngữ cảnh** — một hệ thống cảnh báo tốt phải biết ngữ cảnh (hoặc dùng model dự báo có feature khuyến mãi/lễ rồi cảnh báo trên phần dư — chương 25).

## 6. Template model: phát hiện bất thường

| Phương pháp | Ý tưởng | Mạnh | Yếu |
|---|---|---|---|
| Robust z / IQR | lệch khỏi trung vị | đơn giản, diễn giải được | đơn biến |
| Gaussian / Mahalanobis | mật độ thấp dưới phân phối chuẩn | xác suất, bắt tương quan | giả định một cụm Gaussian |
| Isolation Forest | dễ bị cô lập | nhanh, nhiều chiều, ổn định | bất thường cục bộ |
| LOF | mật độ thấp hơn láng giềng | bất thường cục bộ, hình dạng phức tạp | chậm khi dữ liệu lớn, chọn k, nhiều chiều |
| One-Class SVM | ranh giới kernel | hình dạng linh hoạt | nhạy tham số, $O(n^2)$ |
| Sai số tái tạo (PCA/autoencoder) | không tuân theo cấu trúc ít chiều | ảnh, tín hiệu tương quan | vô dụng khi dữ liệu không có cấu trúc ít chiều |
| Có giám sát | học trực tiếp bất thường | tốt nhất cho kiểu đã có đủ nhãn | không bắt kiểu bất thường mới |

**Thực hành:** feature engineering (tỉ số, so với lịch sử của chính thực thể, ngữ cảnh thời gian) thường quan trọng hơn chọn thuật toán; làm việc với **điểm số** + ngưỡng theo năng lực xử lý (top-k); vòng phản hồi chuyên gia → nhãn; theo dõi drift (bất thường hôm nay có thể là bình thường mới).

## 🧠 Kiểm tra nhanh

1. Phân biệt outlier detection (không giám sát) và novelty detection (bán giám sát).
2. Vì sao model có giám sát bỏ sót kiểu gian lận C, dù được train với nhiều nhãn?
3. Khi nào sai số tái tạo PCA là detector tốt, khi nào vô dụng?
4. Vì sao so với **trung vị** 4 ngày cùng thứ thay vì chỉ tuần trước? Vì sao `.shift(1)`?

## 📌 Tóm tắt

- Có giám sát: mạnh với kiểu đã biết, mù với kiểu mới; novelty/không giám sát: bắt "khác bình thường" — chạy song song.
- Ít nhãn → model có giám sát kém và không ổn định; đánh giá với vài chục nhãn rất nhiễu.
- Sai số tái tạo cần dữ liệu bình thường có cấu trúc ít chiều; kết hợp theo thứ hạng = phòng rủi ro khi không có nhãn.
- Chuỗi thời gian: so với ngữ cảnh quá khứ (robust, không rò rỉ), loại trừ ngữ cảnh đã biết.

## 📚 Tài liệu tham khảo

- Aggarwal (2017) — *Outlier Analysis* (2nd ed.)
- Campos et al. (2016) — *On the evaluation of unsupervised outlier detection*
- scikit-learn User Guide — Novelty and Outlier Detection